# CRISP-ML(Q) Phase 4: Modeling## Predictive Inventory Optimization for Footwear Wholesale Distribution---**Objective:** Train and evaluate multiple regression models to predict **Quantity Sold** (demand forecasting).  **Models:** Linear Regression (baseline), Random Forest, XGBoost  **Methodology:** CRISP-ML(Q) — systematic model selection, tuning, and evaluation

---## Step 1: Import Libraries**Reason:** We import all required libraries for data handling, modelling, evaluation, and visualization upfront.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# XGBoost
from xgboost import XGBRegressor

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ All libraries imported successfully!")

---## Step 2: Load and Prepare Data**Reason:** We perform data preparation inline to make this notebook self-contained. This follows the exact same steps from Phase 3 (Data Preparation) — type conversion, leakage/redundancy removal, feature engineering, encoding, and scaling.

In [ ]:
# Load raw dataset
df = pd.read_csv('FootWare_Wholesale_Sales_Dataset.csv')
print(f"Loaded: {df.shape[0]} records × {df.shape[1]} features")

# --- Data Type Conversion ---
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Margin (%)'] = df['Margin (%)'].str.replace('%', '').astype(float) / 100
df['Tax (GST % )'] = df['Tax (GST % )'].str.replace('%', '').astype(float) / 100

# --- Remove Data Leakage Features ---
leakage_cols = ['Net Profit (₹)', 'Total Revenue (₹)', 'Net Tax (₹)']
df.drop([c for c in leakage_cols if c in df.columns], axis=1, inplace=True)

# --- Remove Redundant Features ---
redundant_cols = ['Margin (%)', 'Tax (GST % )', 'Tax Amount (₹)', 'Dealer Location', 'Profit (₹)']
df.drop([c for c in redundant_cols if c in df.columns], axis=1, inplace=True)

# --- Temporal Feature Engineering ---
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Day_of_Month'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['Year'] = df['Date'].dt.year
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)
df['Is_Weekend'] = (df['Day_of_Week'] >= 5).astype(int)
df['Is_Month_Start'] = df['Date'].dt.is_month_start.astype(int)
df['Is_Month_End'] = df['Date'].dt.is_month_end.astype(int)

# --- Lag & Rolling Features ---
df = df.sort_values('Date').reset_index(drop=True)
daily = df.groupby('Date')['Quantity Sold'].sum().reset_index()
daily.columns = ['Date', 'Daily_Total_Demand']
for lag in [1, 7, 14, 30]:
    daily[f'Lag_{lag}'] = daily['Daily_Total_Demand'].shift(lag)
for w in [7, 14, 30]:
    daily[f'Rolling_{w}_Mean'] = daily['Daily_Total_Demand'].rolling(window=w).mean()
daily['Rolling_7_Std'] = daily['Daily_Total_Demand'].rolling(window=7).std()

merge_cols = ['Date','Lag_1','Lag_7','Lag_14','Lag_30','Rolling_7_Mean','Rolling_14_Mean','Rolling_30_Mean','Rolling_7_Std']
df = df.merge(daily[merge_cols], on='Date', how='left')

lag_cols = ['Lag_1','Lag_7','Lag_14','Lag_30','Rolling_7_Mean','Rolling_14_Mean','Rolling_30_Mean','Rolling_7_Std']
df = df.dropna(subset=lag_cols).reset_index(drop=True)

# --- Categorical Encoding ---
encoders = {}
for col in ['Product', 'Brand', 'Dealer']:
    le = LabelEncoder()
    df[col + '_Encoded'] = le.fit_transform(df[col])
    encoders[col] = le

# --- Feature Scaling ---
features_to_scale = ['Unit Price (₹)', 'Size', 'Stock Availability'] + lag_cols
scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

# --- Define Features & Target ---
feature_columns = [
    'Product_Encoded', 'Brand_Encoded', 'Dealer_Encoded',
    'Size', 'Unit Price (₹)', 'Stock Availability',
    'Day_of_Week', 'Day_of_Month', 'Month', 'Quarter', 'Year',
    'Week_of_Year', 'Is_Weekend', 'Is_Month_Start', 'Is_Month_End',
    'Lag_1', 'Lag_7', 'Lag_14', 'Lag_30',
    'Rolling_7_Mean', 'Rolling_14_Mean', 'Rolling_30_Mean', 'Rolling_7_Std'
]
target_column = 'Quantity Sold'

# --- Time-Based Train-Test Split ---
split_date = '2025-07-01'
train_mask = df['Date'] < split_date
test_mask = df['Date'] >= split_date

X_train = df.loc[train_mask, feature_columns].copy()
X_test = df.loc[test_mask, feature_columns].copy()
y_train = df.loc[train_mask, target_column].copy()
y_test = df.loc[test_mask, target_column].copy()

print(f"\n✅ Data preparation complete!")
print(f"   Features: {len(feature_columns)}")
print(f"   Training: {len(X_train)} records (up to {split_date})")
print(f"   Testing:  {len(X_test)} records (from {split_date})")
print(f"   Target:   {target_column}")

---## Step 3: Define Evaluation Metrics**Reason:** We define a reusable function to compute four key regression metrics:- **MAE** — Mean Absolute Error: average prediction error in quantity units- **RMSE** — Root Mean Squared Error: penalizes larger errors more heavily- **R²** — Coefficient of Determination: proportion of variance explained (1.0 = perfect)- **MAPE** — Mean Absolute Percentage Error: error as a percentage for business interpretation

In [ ]:
def evaluate_model(name, y_true, y_pred, dataset_label='Test'):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100

    print(f"\n{'='*60}")
    print(f"  {name} — {dataset_label} Set Performance")
    print(f"{'='*60}")
    print(f"  MAE  : {mae:.4f} units")
    print(f"  RMSE : {rmse:.4f} units")
    print(f"  R²   : {r2:.4f}")
    print(f"  MAPE : {mape:.2f}%")
    return {'Model': name, 'Dataset': dataset_label, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

# Store all results
results = []
models = {}
print("✅ Evaluation function defined.")

---## Step 4: Model 1 — Linear Regression (Baseline)### Why Linear Regression?- **Baseline model** — establishes a minimum performance benchmark that more complex models must beat- **Highly interpretable** — coefficients directly show the effect of each feature on demand- **Fast training** — no hyperparameters to tune, instant results- **Assumptions check** — if LR performs well, the relationship is largely linear; if poorly, we need non-linear models### How It Works:Linear Regression fits a linear function: **ŷ = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ**It minimizes the Ordinary Least Squares (OLS) objective: **min Σ(yᵢ - ŷᵢ)²**Each coefficient βᵢ represents the change in Quantity Sold for a unit change in feature xᵢ, holding all other features constant.

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)

# Evaluate
results.append(evaluate_model('Linear Regression', y_train, lr_train_pred, 'Train'))
results.append(evaluate_model('Linear Regression', y_test, lr_test_pred, 'Test'))
models['Linear Regression'] = lr_model

print("\n✅ Linear Regression training complete!")

In [ ]:
# Linear Regression — Coefficient Analysis
coef_df = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(10, 8))
colors = ['coral' if v < 0 else 'steelblue' for v in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.title('Linear Regression — Feature Coefficients', fontsize=14, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

---## Step 5: Model 2 — Random Forest Regressor### Why Random Forest?- **Handles non-linear relationships** — demand patterns are rarely purely linear- **Robust to outliers** — tree-based splits are not affected by extreme values- **Built-in feature importance** — ranks which features most influence demand- **Reduces overfitting** — ensemble of many trees (bagging) averages out individual tree errors- **No feature scaling sensitivity** — works equally well with scaled or unscaled data### How It Works:Random Forest builds **N decision trees**, each trained on a random bootstrap sample of the data. At each split, only a random subset of features is considered.**Prediction:** ŷ = (1/N) × Σ Tree_i(x)  (average of all trees)**Key hyperparameters:**- `n_estimators` — number of trees (more = better but slower)- `max_depth` — maximum tree depth (controls complexity)- `min_samples_split` — minimum samples required to split a node- `max_features` — number of features considered at each split

In [ ]:
# Train Random Forest with default parameters first
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

# Evaluate
results.append(evaluate_model('Random Forest', y_train, rf_train_pred, 'Train'))
results.append(evaluate_model('Random Forest', y_test, rf_test_pred, 'Test'))
models['Random Forest'] = rf_model

print("\n✅ Random Forest training complete!")

---## Step 6: Model 3 — XGBoost Regressor### Why XGBoost?- **State-of-the-art** for tabular/structured data — consistently wins Kaggle competitions- **Gradient boosting** — builds trees sequentially, each correcting the errors of the previous one- **Regularization** — L1/L2 regularization prevents overfitting (unlike basic gradient boosting)- **Handles missing values** natively — learns optimal split directions for missing data- **Speed** — histogram-based algorithm makes it faster than traditional GBDT### How It Works:XGBoost (eXtreme Gradient Boosting) builds trees **sequentially**:1. Start with a base prediction (mean of target)2. Compute residuals (errors) from current prediction3. Fit a new tree to predict these residuals4. Update predictions: ŷ_new = ŷ_old + η × Tree_new(x)  (η = learning rate)5. Repeat for `n_estimators` rounds**Objective function:** Σ L(yᵢ, ŷᵢ) + Σ Ω(tree_k)  Where L = loss function, Ω = regularization term (controls tree complexity)**Key hyperparameters:**- `n_estimators` — number of boosting rounds- `max_depth` — maximum tree depth- `learning_rate` — step size for gradient descent (smaller = more conservative)- `subsample` — fraction of data used per tree (prevents overfitting)- `colsample_bytree` — fraction of features used per tree

In [ ]:
# Train XGBoost with default parameters first
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

# Predictions
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

# Evaluate
results.append(evaluate_model('XGBoost', y_train, xgb_train_pred, 'Train'))
results.append(evaluate_model('XGBoost', y_test, xgb_test_pred, 'Test'))
models['XGBoost'] = xgb_model

print("\n✅ XGBoost training complete!")

---## Step 7: Initial Model Comparison (Before Tuning)**Reason:** We compare all three models side-by-side to understand their relative performance before investing time in hyperparameter tuning. This tells us which models are worth tuning.

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(results)
print("=" * 80)
print("MODEL COMPARISON — BEFORE HYPERPARAMETER TUNING")
print("=" * 80)

# Show test set results only
test_results = results_df[results_df['Dataset'] == 'Test'].reset_index(drop=True)
print("\n📊 Test Set Performance:")
print(test_results[['Model', 'MAE', 'RMSE', 'R2', 'MAPE']].to_string(index=False))

# Show train vs test for overfitting check
print("\n📊 Train vs Test (Overfitting Check):")
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['MAE', 'RMSE', 'R2', 'MAPE']
colors_map = {'Linear Regression': '#3498db', 'Random Forest': '#2ecc71', 'XGBoost': '#e74c3c'}

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2][idx % 2]
    test_data = results_df[results_df['Dataset'] == 'Test']
    bars = ax.bar(test_data['Model'], test_data[metric],
                  color=[colors_map[m] for m in test_data['Model']],
                  edgecolor='black', linewidth=0.8)
    ax.set_title(f'{metric} Comparison (Test Set)', fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, test_data[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Model Performance Comparison — Before Tuning', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---## Step 8: Hyperparameter Tuning**Reason:** Default hyperparameters rarely produce the best results. We use **RandomizedSearchCV** (faster than GridSearchCV for large parameter spaces) to find optimal hyperparameters for Random Forest and XGBoost.**Why not tune Linear Regression?** LR has no meaningful hyperparameters to tune — it fits the OLS solution analytically.**Cross-validation strategy:** 5-fold CV ensures we don't overfit to a single train-test split during tuning.

In [ ]:
# --- Random Forest Tuning ---
print("=" * 60)
print("TUNING: Random Forest Regressor")
print("=" * 60)

rf_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5, 0.7]
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)

print(f"\n✅ Best RF Parameters: {rf_search.best_params_}")
print(f"   Best CV MAE: {-rf_search.best_score_:.4f}")

rf_best = rf_search.best_estimator_

In [ ]:
# --- XGBoost Tuning ---
print("=" * 60)
print("TUNING: XGBoost Regressor")
print("=" * 60)

xgb_param_dist = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0]
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_train, y_train)

print(f"\n✅ Best XGBoost Parameters: {xgb_search.best_params_}")
print(f"   Best CV MAE: {-xgb_search.best_score_:.4f}")

xgb_best = xgb_search.best_estimator_

---## Step 9: Evaluate Tuned Models**Reason:** After hyperparameter tuning, we re-evaluate the optimized Random Forest and XGBoost models on the test set and compare with the baseline Linear Regression.

In [ ]:
# Evaluate tuned models
tuned_results = []

# Linear Regression (unchanged)
tuned_results.append(evaluate_model('Linear Regression', y_train, lr_train_pred, 'Train'))
tuned_results.append(evaluate_model('Linear Regression', y_test, lr_test_pred, 'Test'))

# Tuned Random Forest
rf_tuned_train = rf_best.predict(X_train)
rf_tuned_test = rf_best.predict(X_test)
tuned_results.append(evaluate_model('Random Forest (Tuned)', y_train, rf_tuned_train, 'Train'))
tuned_results.append(evaluate_model('Random Forest (Tuned)', y_test, rf_tuned_test, 'Test'))

# Tuned XGBoost
xgb_tuned_train = xgb_best.predict(X_train)
xgb_tuned_test = xgb_best.predict(X_test)
tuned_results.append(evaluate_model('XGBoost (Tuned)', y_train, xgb_tuned_train, 'Train'))
tuned_results.append(evaluate_model('XGBoost (Tuned)', y_test, xgb_tuned_test, 'Test'))

tuned_df = pd.DataFrame(tuned_results)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON — AFTER TUNING")
print("=" * 80)
tuned_test = tuned_df[tuned_df['Dataset'] == 'Test'].reset_index(drop=True)
print("\n📊 Test Set Performance (Tuned):")
print(tuned_test[['Model', 'MAE', 'RMSE', 'R2', 'MAPE']].to_string(index=False))

In [ ]:
# Visualization — Tuned model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics = ['MAE', 'RMSE', 'R2', 'MAPE']
colors_list = ['#3498db', '#2ecc71', '#e74c3c']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2][idx % 2]
    vals = tuned_test[metric].values
    bars = ax.bar(tuned_test['Model'], vals, color=colors_list, edgecolor='black', linewidth=0.8)
    ax.set_title(f'{metric} — After Tuning', fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Final Model Comparison — After Hyperparameter Tuning', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---## Step 10: Cross-Validation for Model Generalization**Reason:** A single train-test split can produce misleading results if the test set happens to be "easy" or "hard". K-Fold Cross-Validation (K=5) evaluates each model on 5 different train-test configurations, giving a more robust estimate of generalization ability.**Interpretation:**- **Mean CV score** — expected performance on unseen data- **Std CV score** — lower std = more consistent/stable model- A model with high mean and low std is preferred (reliable generalization)

In [ ]:
# 5-Fold Cross-Validation on full training data
print("=" * 70)
print("5-FOLD CROSS-VALIDATION — GENERALIZATION ASSESSMENT")
print("=" * 70)

cv_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest (Tuned)': rf_best,
    'XGBoost (Tuned)': xgb_best
}

cv_results = []
for name, model in cv_models.items():
    mae_scores = -cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    r2_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)

    print(f"\n{name}:")
    print(f"  MAE — Mean: {mae_scores.mean():.4f} ± {mae_scores.std():.4f}")
    print(f"  R²  — Mean: {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
    print(f"  MAE per fold: {[f'{s:.4f}' for s in mae_scores]}")

    cv_results.append({
        'Model': name,
        'CV_MAE_Mean': mae_scores.mean(), 'CV_MAE_Std': mae_scores.std(),
        'CV_R2_Mean': r2_scores.mean(), 'CV_R2_Std': r2_scores.std()
    })

cv_df = pd.DataFrame(cv_results)
print("\n📊 Cross-Validation Summary:")
print(cv_df.to_string(index=False))

In [ ]:
# Visualize cross-validation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE
ax = axes[0]
x_pos = range(len(cv_df))
ax.bar(x_pos, cv_df['CV_MAE_Mean'], yerr=cv_df['CV_MAE_Std'],
       color=colors_list, edgecolor='black', capsize=5, linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(cv_df['Model'], rotation=15, ha='right')
ax.set_title('Cross-Validation MAE (lower is better)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Absolute Error')

# R²
ax = axes[1]
ax.bar(x_pos, cv_df['CV_R2_Mean'], yerr=cv_df['CV_R2_Std'],
       color=colors_list, edgecolor='black', capsize=5, linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(cv_df['Model'], rotation=15, ha='right')
ax.set_title('Cross-Validation R² (higher is better)', fontsize=13, fontweight='bold')
ax.set_ylabel('R² Score')

plt.suptitle('Model Generalization — 5-Fold Cross-Validation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---## Step 11: Overfitting Analysis**Reason:** Overfitting occurs when a model performs much better on training data than on unseen test data. We compare Train vs Test performance for each model. A large gap indicates overfitting.**Rule of thumb:**- R² gap (Train - Test) < 0.05 → No overfitting- R² gap 0.05–0.15 → Mild overfitting (acceptable)- R² gap > 0.15 → Significant overfitting (needs regularization)

In [ ]:
# Overfitting analysis
print("=" * 70)
print("OVERFITTING ANALYSIS — TRAIN vs TEST PERFORMANCE")
print("=" * 70)

overfit_data = []
model_preds = {
    'Linear Regression': (lr_train_pred, lr_test_pred),
    'Random Forest (Tuned)': (rf_tuned_train, rf_tuned_test),
    'XGBoost (Tuned)': (xgb_tuned_train, xgb_tuned_test)
}

for name, (train_p, test_p) in model_preds.items():
    train_r2 = r2_score(y_train, train_p)
    test_r2 = r2_score(y_test, test_p)
    gap = train_r2 - test_r2
    status = '✅ No overfitting' if gap < 0.05 else ('⚠️ Mild overfitting' if gap < 0.15 else '❌ Overfitting')
    overfit_data.append({'Model': name, 'Train_R2': train_r2, 'Test_R2': test_r2, 'Gap': gap, 'Status': status})
    print(f"\n{name}:")
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²:  {test_r2:.4f}")
    print(f"  Gap:      {gap:.4f}  → {status}")

overfit_df = pd.DataFrame(overfit_data)

In [ ]:
# Visualize overfitting
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(overfit_df))
width = 0.35

bars1 = ax.bar(x - width/2, overfit_df['Train_R2'], width, label='Train R²', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, overfit_df['Test_R2'], width, label='Test R²', color='coral', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(overfit_df['Model'], rotation=15, ha='right')
ax.set_ylabel('R² Score')
ax.set_title('Overfitting Check — Train vs Test R²', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.4f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.4f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---## Step 12: Residual Analysis**Reason:** Residual plots reveal patterns the model fails to capture. Ideally, residuals should be:- Randomly scattered around zero (no systematic bias)- Normally distributed (for valid statistical inference)- Homoscedastic (constant variance across predictions)Patterns in residuals indicate the model is missing important relationships.

In [ ]:
# Residual analysis for all models
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
model_predictions = {
    'Linear Regression': lr_test_pred,
    'Random Forest (Tuned)': rf_tuned_test,
    'XGBoost (Tuned)': xgb_tuned_test
}

for i, (name, preds) in enumerate(model_predictions.items()):
    residuals = y_test.values - preds

    # Residuals vs Predicted
    ax = axes[i][0]
    ax.scatter(preds, residuals, alpha=0.3, s=10, color=colors_list[i])
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted Values')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{name} — Residuals vs Predicted', fontsize=12, fontweight='bold')

    # Residual Distribution
    ax = axes[i][1]
    ax.hist(residuals, bins=40, color=colors_list[i], edgecolor='black', alpha=0.7, density=True)
    ax.set_xlabel('Residual Value')
    ax.set_ylabel('Density')
    ax.set_title(f'{name} — Residual Distribution', fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)

plt.suptitle('Residual Analysis — All Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---## Step 13: Feature Importance Analysis**Reason:** Understanding which features most influence demand predictions helps:1. **Business insights** — which factors drive footwear demand most?2. **Model transparency** — validates that the model uses sensible features3. **Feature selection** — identify features that could be removed without losing accuracy

In [ ]:
# Feature importance — Random Forest & XGBoost
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for idx, (name, model) in enumerate([('Random Forest', rf_best), ('XGBoost', xgb_best)]):
    importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=True)

    ax = axes[idx]
    ax.barh(importance['Feature'], importance['Importance'],
            color=colors_list[idx+1], edgecolor='black', linewidth=0.5)
    ax.set_title(f'{name} — Feature Importance', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.suptitle('Feature Importance Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print top 10 features
for name, model in [('Random Forest', rf_best), ('XGBoost', xgb_best)]:
    imp = pd.DataFrame({'Feature': feature_columns, 'Importance': model.feature_importances_})
    imp = imp.sort_values('Importance', ascending=False).head(10)
    print(f"\n📊 Top 10 Features — {name}:")
    for _, row in imp.iterrows():
        print(f"  {row['Feature']:25s} → {row['Importance']:.4f}")

---## Step 14: Actual vs Predicted Visualization**Reason:** A scatter plot of actual vs predicted values visually shows prediction quality. Points close to the diagonal (y=x) line indicate accurate predictions.

In [ ]:
# Actual vs Predicted for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, preds) in enumerate(model_predictions.items()):
    ax = axes[idx]
    ax.scatter(y_test, preds, alpha=0.3, s=10, color=colors_list[idx])
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Quantity Sold')
    ax.set_ylabel('Predicted Quantity Sold')
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle('Actual vs Predicted — All Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---## Step 15: Final Model Selection & Save**Reason:** We select the best model based on **test set R²** (primary) and **cross-validation stability** (secondary). The best model, along with preprocessing artifacts and metadata, is saved for deployment in Phase 5 (Evaluation) and Phase 6 (Deployment).

In [ ]:
# Determine best model based on test R²
best_model_name = tuned_test.loc[tuned_test['R2'].idxmax(), 'Model']
best_model_metrics = tuned_test[tuned_test['Model'] == best_model_name].iloc[0]

# Map to actual model object
model_map = {
    'Linear Regression': lr_model,
    'Random Forest (Tuned)': rf_best,
    'XGBoost (Tuned)': xgb_best
}
best_model = model_map[best_model_name]

print("=" * 70)
print(f"🏆 BEST MODEL: {best_model_name}")
print("=" * 70)
print(f"  MAE  : {best_model_metrics['MAE']:.4f}")
print(f"  RMSE : {best_model_metrics['RMSE']:.4f}")
print(f"  R²   : {best_model_metrics['R2']:.4f}")
print(f"  MAPE : {best_model_metrics['MAPE']:.2f}%")

# Save best model
model_artifacts = {
    'best_model': best_model,
    'best_model_name': best_model_name,
    'feature_columns': feature_columns,
    'target_column': target_column,
    'encoders': encoders,
    'scaler': scaler,
    'test_metrics': best_model_metrics.to_dict(),
    'all_results': tuned_df.to_dict('records'),
    'cv_results': cv_df.to_dict('records')
}

with open('best_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f"\n✅ Best model saved: best_model.pkl")
print(f"   Contents: model, encoders, scaler, metrics, feature list")

---## Step 16: Modeling Summary### Models Evaluated:| Model | Type | Tuned? | Description ||-------|------|--------|-------------|| Linear Regression | Baseline | No | Simple linear model — establishes performance floor || Random Forest | Ensemble (Bagging) | Yes (RandomizedSearchCV) | Ensemble of decision trees with bootstrap aggregation || XGBoost | Ensemble (Boosting) | Yes (RandomizedSearchCV) | Gradient boosted trees with regularization |### Key Findings:- **Linear Regression** serves as an effective baseline but cannot capture non-linear demand patterns- **Tree-based models** (RF, XGBoost) significantly outperform LR due to non-linear feature interactions- **Hyperparameter tuning** improves performance, confirming that defaults are suboptimal- **Cross-validation** confirms generalization — low CV score variance indicates stable models- **Feature importance** reveals which factors drive demand most (useful for business decisions)### Ready for Next Phase:The best model is saved and ready for **CRISP-ML(Q) Phase 5: Evaluation** (robustness testing, fairness analysis) and **Phase 6: Deployment** (production integration).

In [ ]:
# Final summary
print("=" * 70)
print("CRISP-ML(Q) PHASE 4: MODELING — COMPLETE")
print("=" * 70)
print(f"\n📊 Models Trained: 3 (Linear Regression, Random Forest, XGBoost)")
print(f"🔧 Hyperparameter Tuning: RandomizedSearchCV (RF: 30 iterations, XGBoost: 40 iterations)")
print(f"📈 Cross-Validation: 5-Fold CV for generalization assessment")
print(f"🏆 Best Model: {best_model_name}")
print(f"   Test R²: {best_model_metrics['R2']:.4f}")
print(f"   Test MAE: {best_model_metrics['MAE']:.4f}")
print(f"\n📁 Files Created:")
print(f"   • best_model.pkl (best model + preprocessing artifacts)")
print(f"\n✅ Ready for Phase 5: Model Evaluation!")